# Data Engineer Agent Tests

Use this notebook to validate the Data Engineer agent end to end: question -> SQL -> pandas DataFrame.

Before running the notebook, make sure:
- `data/orders.db` exists
- `GOOGLE_API_KEY` is available in the environment used by the notebook kernel

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path("..").resolve()
dotenv_path = project_root / ".env"

if not project_root.exists():
    raise FileNotFoundError(
        "Could not find the AskData project root from the current notebook working directory."
    )

if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

load_dotenv(dotenv_path=dotenv_path, override=True)

print(f"Project root: {project_root}")
print(f"Loaded environment from: {dotenv_path}")
print(f"GOOGLE_API_KEY loaded: {bool(os.getenv('GOOGLE_API_KEY'))}")

project_root

Project root: /Users/zac_burns/Documents/Personal/AskData
Loaded environment from: /Users/zac_burns/Documents/Personal/AskData/.env
GOOGLE_API_KEY loaded: True


PosixPath('/Users/zac_burns/Documents/Personal/AskData')

In [15]:
import google.genai as genai
import pandas as pd
from google.genai import types

from askdata.agents import AgentConfig, DataEngineerAgent
from askdata.storage import SQLiteDatabase

In [19]:
MODEL = "gemini-3.1-flash-lite"

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

response = client.models.generate_content(
    model=MODEL,
    contents="What is 2 + 2? Show your reasoning.",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            include_thoughts=True,
        ),
    ),
)

print("=== Raw response.candidates[0].content.parts ===")
for i, part in enumerate(response.candidates[0].content.parts):
    print(f"\n--- Part {i} ---")
    print(f"  thought: {part.thought}")
    print(f"  text:    {part.text!r}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


=== Raw response.candidates[0].content.parts ===

--- Part 0 ---
  thought: None
  text:    '2 + 2 = 4.\n\n**Reasoning:**\n\n1.  **Counting:** If you have a set of two objects (e.g., two apples) and you add another set of two objects, you can count the total by starting at two and counting two more: "three, four."\n2.  **Number Line:** On a number line, if you start at the number 2 and move 2 units to the right, you land on 4.\n3.  **Basic Addition Axioms:** In standard arithmetic (Peano axioms), 2 is defined as the successor of 1, and 3 is the successor of 2. Adding 2 is equivalent to moving to the successor of the successor. Therefore, the successor of 3 is 4.'


In [3]:
database = SQLiteDatabase(project_root / "data" / "askdata.db")
database.ensure_exists()

print(f"Database path: {database.db_path}")
print("Available tables:", database.list_tables())
print()

schema = database.get_schema_summary()

# ✅ After - skip lines that don't match the expected format
for line in schema.split("\n"):
    if not line.strip():
        continue

    parts = line.split(": ", 1)
    if len(parts) != 2:  # <-- guard against malformed/header lines
        continue

    table, columns = parts
    print(f"\n{table}")
    print("-" * 40)
    for col in columns.split(", "):
        print(f"  • {col}")

Database path: /Users/zac_burns/Documents/Personal/AskData/data/askdata.db
Available tables: ['customers', 'geolocation', 'order_items', 'orders', 'payments', 'products', 'sellers', 'walmart_trustpilot_reviews']


- customers
----------------------------------------
  • customer_id TEXT
  • customer_unique_id TEXT
  • customer_zip_code_prefix INTEGER
  • customer_city TEXT
  • customer_state TEXT

- geolocation
----------------------------------------
  • geolocation_zip_code_prefix INTEGER
  • geolocation_lat REAL
  • geolocation_lng REAL
  • geolocation_city TEXT
  • geolocation_state TEXT

- order_items
----------------------------------------
  • order_id TEXT
  • order_item_id INTEGER
  • product_id TEXT
  • seller_id TEXT
  • shipping_limit_date TEXT
  • price REAL
  • freight_value REAL

- orders
----------------------------------------
  • order_id TEXT
  • customer_id TEXT
  • order_status TEXT
  • order_purchase_timestamp TEXT
  • order_approved_at TEXT
  • order_delivered_ca

In [4]:
load_dotenv(override=True)

if not os.getenv("GOOGLE_API_KEY"):
    raise EnvironmentError(
        "GOOGLE_API_KEY is not set in the notebook environment or .env file."
    )

agent = DataEngineerAgent(config=AgentConfig(thinking_level="high", debug=True))
agent

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [5]:
question = "How many orders do we have by status? Order by the most frequent"
result = agent.run(question)

print("Question:")
print(result.question)
print()
print("Generated SQL:")
print(result.sql)
print()
result.dataframe

Question:
How many orders do we have by status? Order by the most frequent

Generated SQL:
SELECT order_status, COUNT(order_id) AS order_count FROM orders GROUP BY order_status ORDER BY order_count DESC



,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [11]:
def ask(question: str, debug: bool = False) -> pd.DataFrame:
    result = agent.run(question)

    sep = "─" * 60

    if debug:
        thinking = next(
            (s.payload for s in result.trace if s.name == "thinking_tokens"), None
        )
        if thinking:
            print(f"{'💭 Thinking':^60}")
            print(sep)
            print(thinking)
            print()

    print(f"{'📝 Generated SQL':^60}")
    print(sep)
    print(result.sql)
    print()

    df = result.dataframe
    print(f"{'📊 Results'} — {len(df):,} row{'s' if len(df) != 1 else ''}")
    print(sep)
    return df

In [13]:
ask("Which order statuses have the longest average delivery times?", debug=True)

                      📝 Generated SQL                       
────────────────────────────────────────────────────────────
SELECT order_status, AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)) AS avg_delivery_days FROM orders WHERE order_delivered_customer_date IS NOT NULL AND order_purchase_timestamp IS NOT NULL GROUP BY order_status ORDER BY avg_delivery_days DESC

📊 Results — 2 rows
────────────────────────────────────────────────────────────


,order_status,avg_delivery_days
0,canceled,20.360006
1,delivered,12.558217


In [8]:
ask(
    "What is the top selling product category all time by dollars? Show units sold and Order by month"
)

Generated SQL:
SELECT p."product category", COUNT(oi.order_item_id) AS units_sold, strftime('%Y-%m', o.order_purchase_timestamp) AS order_month FROM order_items oi JOIN products p ON oi.product_id = p.product_id JOIN orders o ON oi.order_id = o.order_id WHERE p."product category" = (SELECT p2."product category" FROM order_items oi2 JOIN products p2 ON oi2.product_id = p2.product_id GROUP BY p2."product category" ORDER BY SUM(oi2.price) DESC LIMIT 1) GROUP BY order_month ORDER BY order_month ASC



,product category,units_sold,order_month
0,HEALTH BEAUTY,3,2016-09
1,HEALTH BEAUTY,48,2016-10
2,HEALTH BEAUTY,85,2017-01
3,HEALTH BEAUTY,166,2017-02
4,HEALTH BEAUTY,211,2017-03
5,HEALTH BEAUTY,189,2017-04
6,HEALTH BEAUTY,290,2017-05
7,HEALTH BEAUTY,260,2017-06
8,HEALTH BEAUTY,316,2017-07
9,HEALTH BEAUTY,360,2017-08


In [21]:
query = """
-- Step 1: Identify the top-selling product category by total revenue
WITH category_sales AS (
    SELECT
        p."product category"               AS product_category,
        SUM(oi.price)                       AS total_sales
    FROM order_items AS oi
    INNER JOIN products AS p
        ON oi.product_id = p.product_id
    WHERE p."product category" IS NOT NULL
    GROUP BY
        p."product category"
),

top_category AS (
    SELECT
        product_category,
        total_sales
    FROM category_sales
    ORDER BY total_sales DESC
    LIMIT 1
)

-- Step 2: Break down the top category's sales by month
SELECT
    tc.product_category                                              AS product_category,
    strftime('%Y-%m', o.order_purchase_timestamp)                    AS sales_month,
    COUNT(DISTINCT oi.order_id)                                      AS num_orders,
    SUM(oi.price)                                                    AS monthly_sales
FROM order_items AS oi
INNER JOIN products AS p
    ON oi.product_id = p.product_id
INNER JOIN orders AS o
    ON oi.order_id = o.order_id
INNER JOIN top_category AS tc
    ON p."product category" = tc.product_category
WHERE o.order_purchase_timestamp IS NOT NULL
GROUP BY
    tc.product_category,
    strftime('%Y-%m', o.order_purchase_timestamp)
ORDER BY
    sales_month ASC;
"""
database.execute_query(query)

,product_category,sales_month,num_orders,monthly_sales
0,HEALTH BEAUTY,2016-09,1,134.97
1,HEALTH BEAUTY,2016-10,44,4552.51
2,HEALTH BEAUTY,2017-01,82,12561.32
3,HEALTH BEAUTY,2017-02,155,22838.79
4,HEALTH BEAUTY,2017-03,193,25995.25
5,HEALTH BEAUTY,2017-04,176,22935.75
6,HEALTH BEAUTY,2017-05,267,46786.02
7,HEALTH BEAUTY,2017-06,239,32029.39
8,HEALTH BEAUTY,2017-07,278,34896.86
9,HEALTH BEAUTY,2017-08,345,49873.90


In [15]:
database.execute_query("select * from order_items limit 5")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
